# Power Plant Output Prediction

**Tabular Regression Project** — Predict electrical energy output of a power plant.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Combined Cycle Power Plant (Kaggle/UCI, ~9,568 rows)
Target: `PE` (net hourly electrical energy output, MW)

## 2 · Project Overview

We predict the **net hourly electrical energy output** (PE) of a combined cycle power plant from ambient conditions: temperature, pressure, humidity, and exhaust vacuum. This clean 4-feature dataset is ideal for learning regression fundamentals.

The data spans 6 years of hourly readings, making it a large, high-quality regression benchmark.

## 3 · Learning Objectives

1. Work with engineering/industrial sensor data.
2. Understand thermodynamic feature relationships.
3. Apply regression to energy prediction.
4. Compare linear vs. non-linear models.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Predict the **electrical energy output** (MW) of a combined cycle power plant given temperature, exhaust vacuum, ambient pressure, and relative humidity.

## 5 · Why This Project Matters

- **Energy output prediction** is critical for grid management.
- Power plants run more efficiently with accurate demand forecasting.
- Clean 4-feature dataset demonstrates that simple features can be very predictive.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: andonians/random-linear-regression or UCI CCPP |
| **Rows** | ~9,568 |
| **Features** | AT (temperature), V (vacuum), AP (pressure), RH (humidity) |
| **Target** | PE (electrical energy output, MW) |

## 7 · Dataset Source and License

- **Source**: UCI Machine Learning Repository, Combined Cycle Power Plant dataset.
- **License**: CC BY 4.0.
- **Note**: Real sensor data from a combined cycle power plant over 6 years.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'PE'
np.random.seed(SEED)

## 11 · Dataset Download

In [ ]:
import kagglehub, glob
try:
    path = kagglehub.dataset_download('andonians/random-linear-regression')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
# Use largest CSV
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')

# Auto-detect target
if TARGET not in df.columns:
    num_cols = df.select_dtypes('number').columns.tolist()
    TARGET_COL = num_cols[-1]
    print(f'PE not found, using: {TARGET_COL}')
else:
    TARGET_COL = TARGET
print(f'Target: {TARGET_COL}')

## 13 · Exploratory Data Analysis

In [ ]:
num_cols = df.select_dtypes('number').columns.tolist()
n = min(4, len(num_cols))
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
if n == 1: axes = [axes]
for ax, col in zip(axes, num_cols[:n]):
    df[col].hist(bins=30, ax=ax, edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET_COL].describe())
print(f'\nSkewness: {df[TARGET_COL].skew():.2f}')

# Correlation
corr = df.select_dtypes('number').corr()[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False)
print(f'\nCorrelation with {TARGET_COL}:')
print(corr)

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
# Drop non-numeric
for col in df.select_dtypes(include='object').columns:
    df = df.drop(columns=[col])
df = df.dropna()
print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.4f}, MAE={baseline_mae:.4f}, R2={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R2: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R2={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R2={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R2={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation

- **Temperature (AT)** is the strongest predictor — higher temps reduce output.
- **Exhaust vacuum (V)** also strongly affects output.
- **Humidity (RH)** and **pressure (AP)** provide secondary signals.
- These thermodynamic relationships are well-understood in engineering.

## 26 · Limitations

- Single power plant — design-specific.
- No maintenance or operational state features.
- Assumes steady-state operation.
- 4 features may miss complex multi-unit interactions.

## 27 · How to Improve

1. Add polynomial/interaction features.
2. Include operational state indicators.
3. Time-based features (hour, season).
4. Multi-plant ensemble.

## 28 · Production Considerations

- Real-time sensor integration.
- Model monitoring for sensor drift.
- Integration with grid dispatch systems.
- Safety constraints on predictions.

## 29 · Common Mistakes

1. Not scaling features for linear models.
2. Ignoring the physical constraints (PE has bounds).
3. Overfitting on this relatively clean dataset.
4. Not considering time-dependence of readings.

## 30 · Mini Challenge

1. Add AT² polynomial feature.
2. Create temperature-humidity interaction.
3. Compare scaled vs unscaled linear regression.
4. Try polynomial regression (degree 2 and 3).

## 31 · Final Summary

- Power plant output is highly predictable from ambient conditions.
- Temperature dominates — a well-known thermodynamic relationship.
- Even simple linear models do well; boosting adds marginal improvement.
- Clean dataset ideal for learning regression fundamentals.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')